# Week 8 Lab: Advanced Data Structures with Generics – Stacks, Queues, and Deques

## Learning Objectives

By the end of this lab, you will be able to:

- Design and implement generic stack, queue, and deque classes using Python's `typing` module.
- Apply abstract data types to simulate real-world systems (e.g., undo/redo mechanisms, expression evaluation, job scheduling).
- Demonstrate deep understanding of time and space complexity implications of different operations.
- Solve complex problems using these core data structures under performance and structural constraints.

## Part 1: Generic Stack, Queue, and Deque

Design the following generic classes using Python’s `typing` module:

1. `GenericStack[T]`
2. `GenericQueue[T]`
3. `GenericDeque[T]`

You are **not** allowed to use Python lists directly for these classes’ core logic. Use `collections.deque` internally.

Your implementation should support:

- Type hinting with `TypeVar`
- All standard operations (push, pop, enqueue, dequeue, etc.)
- Additional utilities: `__str__`, `__len__`, `is_empty()`, etc.


In [ ]:
# GenericStack with Type Hints
from typing import TypeVar, Generic, Optional
from collections import deque

T = TypeVar('T')

class GenericStack(Generic[T]):
    def __init__(self) -> None:
        self._stack = deque()

    def push(self, item: T) -> None:
        self._stack.append(item)

    def pop(self) -> Optional[T]:
        return self._stack.pop() if not self.is_empty() else None

    def peek(self) -> Optional[T]:
        return self._stack[-1] if not self.is_empty() else None

    def is_empty(self) -> bool:
        return len(self._stack) == 0

    def __len__(self) -> int:
        return len(self._stack)

    def __str__(self) -> str:
        return str(list(self._stack))


(similarly define `GenericQueue` and `GenericDeque`)

In [ ]:
# GenericQueue with Type Hints
class GenericQueue(Generic[T]):
    def __init__(self) -> None:
        self._queue = deque()

    def enqueue(self, item: T) -> None:
        self._queue.append(item)

    def dequeue(self) -> Optional[T]:
        return self._queue.popleft() if not self.is_empty() else None

    def peek(self) -> Optional[T]:
        return self._queue[0] if not self.is_empty() else None

    def is_empty(self) -> bool:
        return len(self._queue) == 0

    def __len__(self) -> int:
        return len(self._queue)

    def __str__(self) -> str:
        return str(list(self._queue))


In [ ]:
# GenericDeque with Type Hints
class GenericDeque(Generic[T]):
    def __init__(self) -> None:
        self._deque = deque()

    def append_left(self, item: T) -> None:
        self._deque.appendleft(item)

    def append_right(self, item: T) -> None:
        self._deque.append(item)

    def pop_left(self) -> Optional[T]:
        return self._deque.popleft() if not self.is_empty() else None

    def pop_right(self) -> Optional[T]:
        return self._deque.pop() if not self.is_empty() else None

    def peek_left(self) -> Optional[T]:
        return self._deque[0] if not self.is_empty() else None

    def peek_right(self) -> Optional[T]:
        return self._deque[-1] if not self.is_empty() else None

    def is_empty(self) -> bool:
        return len(self._deque) == 0

    def __len__(self) -> int:
        return len(self._deque)

    def __str__(self) -> str:
        return str(list(self._deque))


## Part 2: Practical Applications

### 2.1 Undo-Redo Stack (Text Editor Simulation)

Implement a class `TextEditor` that internally maintains two stacks for supporting **Undo** and **Redo** operations.

Features:

- `write(text: str)`: Add text.
- `undo()`: Remove last change.
- `redo()`: Reapply last undone change.
- `get_content() -> str`

Use `GenericStack[str]` for both undo and redo functionality.

In [ ]:
# Text Editor with Undo/Redo Stack
class TextEditor:
    def __init__(self) -> None:
        self.content = ""
        self.undo_stack = GenericStack[str]()
        self.redo_stack = GenericStack[str]()

    def write(self, text: str) -> None:
        self.undo_stack.push(self.content)
        self.content += text
        self.redo_stack = GenericStack[str]()  # Clear redo stack

    def undo(self) -> None:
        if not self.undo_stack.is_empty():
            self.redo_stack.push(self.content)
            self.content = self.undo_stack.pop()

    def redo(self) -> None:
        if not self.redo_stack.is_empty():
            self.undo_stack.push(self.content)
            self.content = self.redo_stack.pop()

    def get_content(self) -> str:
        return self.content


### 2.2 Expression Evaluator using Stack

Build a fully-functional **Postfix (Reverse Polish Notation)** expression evaluator that supports:

- Integer operands
- Operators: `+`, `-`, `*`, `/`, `^`

Your task is to:

- Parse the input postfix string
- Use `GenericStack[int]`
- Return the evaluated result

In [ ]:
# Postfix Expression Evaluator
def evaluate_postfix(expression: str) -> int:
    stack = GenericStack[int]()
    tokens = expression.split()

    for token in tokens:
        if token.isdigit() or (token.startswith('-') and token[1:].isdigit()):
            stack.push(int(token))
        else:
            right = stack.pop()
            left = stack.pop()
            if right is None or left is None:
                raise ValueError("Invalid postfix expression")

            if token == '+':
                stack.push(left + right)
            elif token == '-':
                stack.push(left - right)
            elif token == '*':
                stack.push(left * right)
            elif token == '/':
                stack.push(int(left / right))  # Use int to mimic integer division
            elif token == '^':
                stack.push(left ** right)
            else:
                raise ValueError(f"Unknown operator: {token}")

    result = stack.pop()
    if not stack.is_empty():
        raise ValueError("Invalid postfix expression")
    return result


*Example:*  
Input: `"3 4 + 2 * 7 /"`  
Output: `2`

### 2.3 Thread-Safe Task Queue (Advanced)

Simulate a **task scheduler** where tasks are enqueued by one thread and dequeued by another.

Implement:

- `TaskQueue[T]`: A generic queue class that is thread-safe
- Use `threading.Lock` for synchronization

Write a simulation with:

- One producer thread that generates `n` tasks
- One consumer thread that consumes and processes them

Use time delays and logging to simulate load.

In [ ]:
# Thread-Safe TaskQueue and Simulation
from threading import Thread, Lock
import time
import random

class TaskQueue(Generic[T]):
    def __init__(self) -> None:
        self.queue = deque()
        self.lock = Lock()

    def enqueue(self, item: T) -> None:
        with self.lock:
            self.queue.append(item)
            print(f"Enqueued: {item}")

    def dequeue(self) -> Optional[T]:
        with self.lock:
            if self.queue:
                item = self.queue.popleft()
                print(f"Dequeued: {item}")
                return item
            return None

def producer(task_queue: TaskQueue[str]) -> None:
    for i in range(10):
        task = f"Task-{i}"
        task_queue.enqueue(task)
        time.sleep(random.uniform(0.1, 0.4))

def consumer(task_queue: TaskQueue[str]) -> None:
    for _ in range(10):
        task = None
        while task is None:
            task = task_queue.dequeue()
            time.sleep(random.uniform(0.1, 0.4))
        print(f"Processed: {task}")

# Uncomment to run simulation
# tq = TaskQueue[str]()
# t1 = Thread(target=producer, args=(tq,))
# t2 = Thread(target=consumer, args=(tq,))
# t1.start()
# t2.start()
# t1.join()
# t2.join()


### 3.1 Deque-based Sliding Window Maximum

You are given an array `nums` and an integer `k`. Implement a function that returns the **maximum value in every sliding window of size `k`** using a deque.

Constraints:

- Do not use `max()` directly.
- Use `GenericDeque` for optimization.

In [ ]:
# Sliding Window Maximum
from typing import List

def sliding_window_maximum(nums: List[int], k: int) -> List[int]:
    dq = GenericDeque[int]()  # Store indices
    result = []

    for i in range(len(nums)):
        # Remove indices out of the current window
        if not dq.is_empty() and dq.peek_left() < i - k + 1:
            dq.pop_left()

        # Remove smaller elements in k range
        while not dq.is_empty() and nums[dq.peek_right()] < nums[i]:
            dq.pop_right()

        dq.append_right(i)

        # Add to result from k-1 onwards
        if i >= k - 1:
            result.append(nums[dq.peek_left()])
    return result


Example:

```python
Input: nums = [1,3,-1,-3,5,3,6,7], k = 3  
Output: [3,3,5,5,6,7]
```